In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, LogisticRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, classification_report, roc_auc_score
import joblib
import pandas as pd
import numpy as np

# 正规方程线性回归

In [3]:

fe_cal = fetch_california_housing(data_home='data')

print("特征值")
print(fe_cal.data.shape)
print(fe_cal.data[0])
print("目标值")
print(fe_cal.target) 
print('-' * 50)
print(fe_cal.DESCR)
print('-' * 50)
print(fe_cal.feature_names) 

获取特征值
(20640, 8)
--------------------------------------------------
[   8.3252       41.            6.98412698    1.02380952  322.
    2.55555556   37.88       -122.23      ]
目标值
[4.526 3.585 3.521 ... 0.923 0.847 0.894]
--------------------------------------------------
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repos

In [7]:
x_train, x_test, y_train, y_test = train_test_split(fe_cal.data, fe_cal.target, test_size=0.25, random_state=1)
print(x_train.shape)
std_x = StandardScaler()

x_train = std_x.fit_transform(x_train) 
x_test = std_x.transform(x_test) 
# 对特征标准化

(15480, 8)


不对目标值进行标准化

In [17]:
import os
lr = LinearRegression()
lr.fit(x_train, y_train)
# 使用正规方程进行线性回归
print('回归系数', lr.coef_)
y_predict = lr.predict(x_test)

print("预测价格：", y_predict.ravel()[0:10])
print("均方误差：", mean_squared_error(y_test, y_predict))
# 均方误差，公式是(y_test-y_predict)^2/n

回归系数 [[ 0.71942632  0.10518431 -0.23147194  0.26802332 -0.00448136 -0.03495117
  -0.7849086  -0.76307353]]
预测价格： [ 0.039975   -0.9856667   0.54595901 -0.31917221  0.65037085  1.23359413
  0.81054876 -0.38917515 -0.28938242 -0.05080248]
均方误差： 4.7705005262352405


对目标值进行标准化

In [22]:
x_train, x_test, y_train, y_test = train_test_split(fe_cal.data, fe_cal.target, test_size=0.25, random_state=1)

print(x_train.shape)

std_x = StandardScaler()
x_train = std_x.fit_transform(x_train)
x_test = std_x.transform(x_test) 
# 对标签值进行标准化

std_y = StandardScaler()
print(y_train.shape)
# 如果要标准化y, 必须将其特征化(变成二维)
y_train = std_y.fit_transform(y_train.reshape(-1, 1))
print(y_train.shape)
print('-'*50)
lr = LinearRegression()
lr.fit(x_train, y_train)
print('回归系数', lr.coef_)
y_predict = lr.predict(x_test)
y_lr_predict = std_y.inverse_transform(y_predict).ravel()
# inverse_transform将标准化后的数据转换成原始数据, ravel将二维数据转换成一维数据
try:
    os.unlink('./tmp/test.pkl') 
    # 删除之前的模型文件
except:
    pass
joblib.dump(lr, "./tmp/test.pkl")
# 保存训练好的模型，模型中保存的是参数和模型结构
print("预测价格：", y_lr_predict[0:10])
print("均方误差：", mean_squared_error(y_test, y_lr_predict))

(15480, 8)
(15480,)
(15480, 1)
--------------------------------------------------
回归系数 [[ 0.71942632  0.10518431 -0.23147194  0.26802332 -0.00448136 -0.03495117
  -0.7849086  -0.76307353]]
预测价格： [2.12391852 0.93825754 2.7088455  1.70873764 2.82954754 3.50376456
 3.0147162  1.62781292 1.74317518 2.01897806]
均方误差： 0.5356532845422556


使用正规方程进行线性回归时, 对目标值标准化没什么意义

练习加载模型

In [25]:
# 模型存储的是目标值标准化的情况
model = joblib.load("./tmp/test.pkl")
y_predict = model.predict(x_test)

print("预测结果：", y_predict.ravel ()[0:10])
y_predict = lr.predict(x_test)
y_lr_predict = std_y.inverse_transform(y_predict).ravel()
print("均方误差：", mean_squared_error(y_test, y_lr_predict))

预测结果： [ 0.039975   -0.9856667   0.54595901 -0.31917221  0.65037085  1.23359413
  0.81054876 -0.38917515 -0.28938242 -0.05080248]
均方误差： 0.5356532845422556


# 梯度下降法

In [31]:
# 最速下降法的原理
w=1  #随机初始值
learning_rate=0.1  #即"学习率", 可以调节，是超参数
def loss(w):  # 损失函数
    return 3*w**2+2*w+2
def diff_loss(w):
    return 6*w+2
for i in range(30):
    w=w-learning_rate*diff_loss(w)
    print(f'w {w} 损失{loss(w)}')

w 0.19999999999999996 损失2.5199999999999996
w -0.12000000000000005 损失1.8032
w -0.24800000000000003 损失1.688512
w -0.2992 损失1.67016192
w -0.31968 损失1.6672259072
w -0.327872 损失1.666756145152
w -0.33114879999999997 损失1.6666809832243201
w -0.33245952 损失1.6666689573158913
w -0.332983808 损失1.6666670331705427
w -0.3331935232 损失1.6666667253072869
w -0.33327740928 损失1.666666676049166
w -0.333310963712 损失1.6666666681678666
w -0.3333243854848 损失1.6666666669068586
w -0.33332975419392 损失1.6666666667050973
w -0.333331901677568 损失1.6666666666728156
w -0.3333327606710272 损失1.6666666666676506
w -0.3333331042684109 损失1.6666666666668242
w -0.33333324170736434 损失1.6666666666666918
w -0.33333329668294576 损失1.6666666666666707
w -0.3333333186731783 损失1.6666666666666674
w -0.3333333274692713 损失1.6666666666666667
w -0.3333333309877085 损失1.6666666666666667
w -0.3333333323950834 损失1.6666666666666665
w -0.33333333295803336 损失1.6666666666666667
w -0.33333333318321334 损失1.6666666666666667
w -0.33333333327328535 损失1.6

In [65]:
x_train, x_test, y_train, y_test = train_test_split(
    fe_cal.data, fe_cal.target, test_size=0.25, random_state=1)

std_x = StandardScaler()
x_train_std = std_x.fit_transform(x_train)
x_test_std = std_x.transform(x_test)

std_y = StandardScaler()
y_train_std = std_y.fit_transform(y_train.reshape(-1,1)).ravel()
# 变成二维标准化后再压平, 因为SGD只接受一维输入
sgd = SGDRegressor(eta0=0.01,penalty='l2', max_iter=1000)
# eta0:初始学习率, penalty:正则化项, max_iter:最大迭代次数
# learning_rate: 学习率 
# 默认为optimal, 近似最优步长
# constant, 固定学习率
# invscaling, 反比例衰减
# adaptive, 初始用eta0, 如果loss不再下降, 则除以5
sgd.fit(x_train_std, y_train_std)
#训练
print('回归系数', sgd.coef_)
# 预测测试集的房子价格
y_predict_std = sgd.predict(x_test_std)
y_predict = std_y.inverse_transform(y_predict_std.reshape(-1,1))
print("预测价格：", y_predict[:10].ravel())
print("均方误差：", mean_squared_error(y_test, y_predict))

回归系数 [ 0.70341687  0.10673132 -0.19795706  0.2331561  -0.01126054 -0.03095282
 -0.79870654 -0.77337904]
预测价格： [2.11981184 0.94475287 2.6974146  1.71288519 2.82741859 3.47440679
 3.03565633 1.62418413 1.75150793 2.03022269]
均方误差： 0.5361985399902117


正则化是在每次计算梯度时, 在误差函数上加上一个正则化项  
限制参数不要太大, 降低复杂度, 减少过拟合  
正则化项有不同的形式  
L1正则化:  
正则化项是所有参数的绝对值之和, 会产生许多稀疏解, 很多参数变为0  
对参数的惩罚是"以恒定力度往0拉", 小参数很容易变成0   
模型只保留少量重要特征, 适合高维情况
L2正则化:  
正则化项是所有参数的平方之和, 会产生权重较小的解, 所有参数都接近0  
对参数的惩罚是"参数越大, 惩罚越大", 会产生许多较小参数, 但通常不为0  
模型使用所有特征, 但权重较小, 对输入变化更不敏感(稳定), 输出变化更平缓, 防止过拟合

# Lasso回归

In [66]:
# Lasso回归是使用L1正则化项的回归, 具体求解方法可以用正规方程也可以用梯度下降
ls = Lasso(alpha=0.001)

ls.fit(x_train_std, y_train_std)

print(ls.coef_)

print(ls.predict(x_test).shape)
print('-'*50)

y_predict_std = ls.predict(x_test_std)
y_predict = std_y.inverse_transform(y_predict_std.reshape(-1,1))

print("均方误差：", mean_squared_error(y_test, y_predict))

[ 0.71431792  0.10613811 -0.21758465  0.25415162 -0.00311065 -0.03403136
 -0.77399969 -0.75154125]
(5160,)
--------------------------------------------------
均方误差： 0.5356382077349954


# 岭回归

In [74]:
# 线性回归加上L2正则项
# Lasso回归是使用L1正则化项的回归, 具体求解方法可以用正规方程也可以用梯度下降
rd = Ridge(alpha=0.001)

rd.fit(x_train_std, y_train_std)

print(rd.coef_)

print(rd.predict(x_test).shape)
print('-'*50)

y_predict_std = rd.predict(x_test_std)
y_predict = std_y.inverse_transform(y_predict_std.reshape(-1,1))

print("均方误差：", mean_squared_error(y_test, y_predict))

[ 0.7194263   0.10518438 -0.23147179  0.26802312 -0.00448134 -0.03495118
 -0.78490787 -0.7630728 ]
(5160,)
--------------------------------------------------
均方误差： 0.5356532762093725


alpha指正则化项的强度  
L1: Loss = Loss + α * sum(abs(w_i))   
L2: Loss = Loss + α * sum(w_i^2)  
α越大, 模型越稀疏/平滑

Elastic Net 回归（结合 L1 + L2 正则化）

In [75]:
from sklearn.linear_model import ElasticNet
model = ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42) 
# 混合L1和L2正则化, 兼顾稀疏性和平滑性
model.fit(x_train_std, y_train_std)
y_pred_std = model.predict(x_test_std)
y_pred = std_y.inverse_transform(y_pred_std.reshape(-1,1))

mse = mean_squared_error(y_test, y_pred)
print("均方误差:", mse)

均方误差: 0.5355884667450996


# 逻辑回归

In [80]:

column = ['Sample code number', 'Clump Thickness', 
          'Uniformity of Cell Size', 'Uniformity of Cell Shape',
          'Marginal Adhesion', 'Single Epithelial Cell Size', 'Bare Nuclei',
          'Bland Chromatin', 'Normal Nucleoli', 'Mitoses', 'Class']

# 读取数据
# data = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data", names=column)
data = pd.read_csv("./data/breast-cancer-wisconsin.csv", names=column)
print(data)
print(data.info())
print('-'*50)
data.describe(include='all')
# 发现Bare Nuclei列包含字符串

     Sample code number  Clump Thickness  Uniformity of Cell Size  \
0               1000025                5                        1   
1               1002945                5                        4   
2               1015425                3                        1   
3               1016277                6                        8   
4               1017023                4                        1   
..                  ...              ...                      ...   
694              776715                3                        1   
695              841769                2                        1   
696              888820                5                       10   
697              897471                4                        8   
698              897471                4                        8   

     Uniformity of Cell Shape  Marginal Adhesion  Single Epithelial Cell Size  \
0                           1                  1                            2   
1        

,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
count,6.990000e+02,699.000000,699.000000,699.000000,699.000000,699.000000,699,699.000000,699.000000,699.000000,699.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,402,NaN,NaN,NaN,NaN
mean,1.071704e+06,4.417740,3.134478,3.207439,2.806867,3.216023,NaN,3.437768,2.866953,1.589413,2.689557
std,6.170957e+05,2.815741,3.051459,2.971913,2.855379,2.214300,NaN,2.438364,3.053634,1.715078,0.951273
min,6.163400e+04,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,1.000000,1.000000,1.000000,2.000000
25%,8.706885e+05,2.000000,1.000000,1.000000,1.000000,2.000000,NaN,2.000000,1.000000,1.000000,2.000000
50%,1.171710e+06,4.000000,1.000000,1.000000,1.000000,2.000000,NaN,3.000000,1.000000,1.000000,2.000000
75%,1.238298e+06,6.000000,5.000000,5.000000,4.000000,4.000000,NaN,5.000000,4.000000,1.000000,4.000000


In [78]:
data['Bare Nuclei'].unique() 

<StringArray>
['1', '10', '2', '4', '3', '9', '7', '?', '5', '8', '6']
Length: 11, dtype: str

In [81]:
data = data.replace(to_replace='?', value=np.nan)
# 将异常值转成NaN, 然后删除整个样本
data = data.dropna()
print(data.shape)
# 还剩下683个样本

(683, 11)


In [87]:
data.info()

<class 'pandas.DataFrame'>
Index: 683 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           683 non-null    int64
 1   Clump Thickness              683 non-null    int64
 2   Uniformity of Cell Size      683 non-null    int64
 3   Uniformity of Cell Shape     683 non-null    int64
 4   Marginal Adhesion            683 non-null    int64
 5   Single Epithelial Cell Size  683 non-null    int64
 6   Bare Nuclei                  683 non-null    str  
 7   Bland Chromatin              683 non-null    int64
 8   Normal Nucleoli              683 non-null    int64
 9   Mitoses                      683 non-null    int64
 10  Class                        683 non-null    int64
dtypes: int64(10), str(1)
memory usage: 64.0 KB


In [89]:
data[column[6]] = data[column[6]].astype('int64')
data.info()
# 将Bare Nuclei列转成int64

<class 'pandas.DataFrame'>
Index: 683 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           683 non-null    int64
 1   Clump Thickness              683 non-null    int64
 2   Uniformity of Cell Size      683 non-null    int64
 3   Uniformity of Cell Shape     683 non-null    int64
 4   Marginal Adhesion            683 non-null    int64
 5   Single Epithelial Cell Size  683 non-null    int64
 6   Bare Nuclei                  683 non-null    int64
 7   Bland Chromatin              683 non-null    int64
 8   Normal Nucleoli              683 non-null    int64
 9   Mitoses                      683 non-null    int64
 10  Class                        683 non-null    int64
dtypes: int64(11)
memory usage: 64.0 KB


In [86]:
data['Class'].unique()
# 分类2和4代表良性和恶性

array([2, 4])

In [90]:
x_train, x_test, y_train, y_test \
    = train_test_split(data.loc[:,column[1:10]], data.loc[:,column[10]], 
                       test_size=0.25, random_state=1)
# 分割训练集和测试集, 取1列到9列作为标签, 10列作为标签列
std = StandardScaler()
x_train = std.fit_transform(x_train) 
x_test = std.transform(x_test)
# 标准化
x_train[0]

array([-1.21629973, -0.70863282, -0.75174943,  0.04301674, -0.55657068,
       -0.71054972, -0.99312055, -0.62911518, -0.36280962])

In [105]:
lg = LogisticRegression(C=0.5, solver='lbfgs')
# C是正则化强度, 默认使用L2正则化
# solver是求解方法:
# liblinear(坐标下降法), 适合小数据集, 支持L1/L2正则化
# 默认是lbfgs拟牛顿法, 适合中小数据集, 支持L2正则化
# saga改进的随机平均梯度下降法, 适合大数据集和稀疏矩阵, 支持L1和L2
# newton-cg牛顿法, 支持L2
lg.fit(x_train, y_train)
# print(lg.coef_)
# print(lg.intercept_)
# 逻辑回归的权重参数和偏置项
y_pred = lg.predict(x_test)
print("准确率：", lg.score(x_test, y_test))
print(y_pred[0:10])
print(y_test[0:10].to_numpy())
print('-'*50)
print(lg.predict_proba(x_test)[0:5])
# 0到4号样本的预测概率, 分别是预测为2的概率和预测为4的概率

准确率： 0.9824561403508771
[2 2 2 4 2 4 2 2 4 4]
[2 2 2 4 2 4 2 2 4 4]
--------------------------------------------------
[[0.94893919 0.05106081]
 [0.99494175 0.00505825]
 [0.98365149 0.01634851]
 [0.02707911 0.97292089]
 [0.99732446 0.00267554]]


In [98]:
print(classification_report(y_test, y_pred, labels=[2, 4], 
                            target_names=["良性", "恶性"]))
print("AUC指标：", roc_auc_score(y_test, y_pred))

              precision    recall  f1-score   support

          良性       0.97      1.00      0.99       111
          恶性       1.00      0.95      0.97        60

    accuracy                           0.98       171
   macro avg       0.99      0.97      0.98       171
weighted avg       0.98      0.98      0.98       171

AUC指标： 0.975
